In [ ]:
import itk
import ipywidgets
from ipywidgets import Label, HBox, VBox
import matplotlib.pyplot as plt
import numpy as np
import os
import base64
import tempfile


def showImage(output, image, title):
    """Show a 2D image, or the center slices of a 3D image, in an Output widget."""
    array = itk.array_view_from_image(image)
    if image.GetImageDimension() == 3:
        # NumPy indexes ITK images as [z, y, x]
        z, y, x = np.array(array.shape) // 2
        sx, sy, sz = itk.spacing(image)
        views = [
            ("axial", array[z, :, :], sy / sx, "upper"),
            ("coronal", array[:, y, :], sz / sx, "lower"),
            ("sagittal", array[:, :, x], sz / sy, "lower"),
        ]
    else:
        sx, sy = itk.spacing(image)
        views = [("", array, sy / sx, "upper")]
    fig, axes = plt.subplots(1, len(views), figsize=(4 * len(views), 4), squeeze=False)
    for ax, (name, imageSlice, aspect, origin) in zip(axes[0], views):
        ax.imshow(imageSlice, cmap="gray", aspect=aspect, origin=origin)
        ax.set_title(f"{title}, {name}" if name else title)
        ax.axis("off")
    fig.tight_layout()
    with output:
        output.clear_output(wait=True)
        display(fig)
    plt.close(fig)


def checkerboard(image1, image2, pattern=5):
    """Checkerboard of two images, with image2 resampled onto the image1 grid."""
    image2 = itk.resample_image_filter(
        image2, use_reference_image=True, reference_image=image1
    )
    return itk.checker_board_image_filter(image1, image2, checker_pattern=pattern)


def readUploadedImage(uploader):
    uploaded = uploader.value[0]
    with tempfile.TemporaryDirectory() as directory:
        # ITK selects the image file format from the file name extension
        fileName = os.path.join(directory, os.path.basename(uploaded["name"]))
        with open(fileName, "wb") as fp:
            fp.write(uploaded["content"])
        return itk.imread(fileName, itk.F)


# these FileUpload widgets have maximum file size
# controlled via jupyter_notebook_config.py
# https://github.com/jupyter-widgets/ipywidgets/issues/2522
# Start with:
#  python -m jupyter notebook --config ./.jupyter/jupyter_notebook_config.py ./examples/ITK_Registration_App.ipynb
# for Jupyter, or
#  python -m voila --Voila.tornado_settings '{"websocket_max_message_size":838860800}' --theme=dark ./examples/ITK_Registration_App.ipynb
# for Voilà on Linux. Command is slightly different on Windows:
#  python -m voila "--Voila.tornado_settings={\"websocket_max_message_size\":838860800}" --theme=dark ./examples/ITK_Registration_App.ipynb

fixedUploader = ipywidgets.FileUpload(accept=".nrrd,.mha,.nii,image/*", multiple=False)
movingUploader = ipywidgets.FileUpload(accept=".nrrd,.mha,.nii,image/*", multiple=False)

fixedOutput = ipywidgets.Output()
movingOutput = ipywidgets.Output()
resultOutput = ipywidgets.Output()

uploaders = HBox(
    [
        VBox([Label("Fixed image"), fixedUploader]),
        VBox([Label("Moving image"), movingUploader]),
    ]
)
display(VBox([uploaders, fixedOutput, movingOutput, resultOutput]))

out = ipywidgets.Output()
display(out)  # for status messages

regType = ipywidgets.RadioButtons(
    options=["rigid", "affine", "bspline"],
    value="rigid",
    description="Type:",
    disabled=False,
)
registerButton = ipywidgets.Button(description="Register")
buttons = HBox([regType, registerButton])
display(buttons)

appState = {"fixed": None, "moving": None, "result": None}


def uploadedImage(b):
    if b["owner"] is fixedUploader:
        key, output = "fixed", fixedOutput
    else:
        key, output = "moving", movingOutput
    out.clear_output()
    # A new upload invalidates the previous image and registration result
    appState[key] = None
    appState["result"] = None
    output.clear_output()
    resultOutput.clear_output()
    buttons.children = (regType, registerButton)
    try:
        appState[key] = readUploadedImage(b["owner"])
        showImage(output, appState[key], key.capitalize())

        if appState["fixed"] is not None and appState["moving"] is not None:
            showImage(
                resultOutput,
                checkerboard(appState["fixed"], appState["moving"]),
                "Fixed/moving checkerboard",
            )
    except Exception as e:
        with out:
            print("Could not show the uploaded image:", e)


fixedUploader.observe(uploadedImage, names="value")
movingUploader.observe(uploadedImage, names="value")


def registerImages(b):
    out.clear_output(wait=True)
    if appState["fixed"] is None or appState["moving"] is None:
        with out:
            print("Upload a fixed and a moving image first.")
        return

    parameters = itk.ParameterObject.New()

    resolutions = 3
    default_rigid = itk.ParameterObject.GetDefaultParameterMap("rigid", resolutions)
    parameters.AddParameterMap(default_rigid)

    if regType.value != "rigid":
        resolutions = 2
        default_affine = itk.ParameterObject.GetDefaultParameterMap(
            "affine", resolutions
        )
        parameters.AddParameterMap(default_affine)

    if regType.value == "bspline":
        resolutions = 1
        default_bspline = itk.ParameterObject.GetDefaultParameterMap(
            "bspline", resolutions
        )
        parameters.AddParameterMap(default_bspline)

    parameters.RemoveParameter("ResultImageFormat")
    with out:
        print("Executing " + regType.value + " registration. Please wait...")

    appState["result"], params = itk.elastix_registration_method(
        appState["fixed"],
        appState["moving"],
        parameter_object=parameters,
        log_to_file=True,
        log_file_name="elx_HASI.log",
        output_directory=".",
    )

    with out:
        print("Compressing result image...")
    itk.imwrite(appState["result"], "./result.nrrd", compression=True)

    showImage(
        resultOutput,
        checkerboard(appState["fixed"], appState["result"]),
        "Fixed/registered checkerboard",
    )

    # create a button which will contain base64-encoded file content
    # that way server permission don't matter for downloading
    with open("./result.nrrd", mode="rb") as file:
        fileContent = file.read()
    fileContent64 = base64.b64encode(fileContent)
    html = """<a download="MovingRegistered.nrrd"
        href="data:image/nrrd;base64,{payload}"
        target="_blank">Download moving registered image</a>"""
    html = html.format(payload=fileContent64.decode())
    downloadLink = ipywidgets.HTML(html)
    buttons.children = (buttons.children[0], buttons.children[1], downloadLink)

    out.clear_output()


registerButton.on_click(registerImages)